# Asistente Fiscal con Gemini, RAG y LangGraph

Agente experto para gestorías españolas. Asesora sobre obligaciones fiscales de **autónomos y sociedades**: cómo rellenar declaraciones, plazos del calendario fiscal y avisos de antelación.

**Stack:** Google Gemini · ChromaDB · LangGraph · LangChain

---

## Índice

1. [Instalación y configuración](#1-instalación-y-configuración)
2. [Carga y procesado de documentos](#2-carga-y-procesado-de-documentos)
3. [Creación de la base de conocimiento vectorial](#3-creación-de-la-base-de-conocimiento-vectorial)
4. [Diseño del agente LangGraph](#4-diseño-del-agente-langgraph)
   - 4a. System prompt con few-shot examples
   - 4b. Grafo con routing condicional y gestión de tokens
5. [Lógica de avisos por antelación](#5-lógica-de-avisos-por-antelación)
6. [Moderación en cascada](#6-moderación-en-cascada)
7. [Herramientas fiscales con @tool y loop ReAct](#7-herramientas-fiscales-con-tool-y-loop-react)
8. [LLM-as-Judge — evaluación de calidad](#8-llm-as-judge--evaluación-de-calidad)
9. [Demo interactiva](#9-demo-interactiva)

---

### Arquitectura del grafo

```
START
  │
  ▼
podar_historial  ← elimina mensajes si historial > MAX_MESSAGES
  │
  ▼
clasificar_consulta  ← detecta tipo: plazos | documentos | general
  │
  ├─► recuperar_plazos     (prioriza CSVs de calendario)
  ├─► recuperar_documentos (prioriza manuales PDF)
  └─► recuperar_general    (mezcla balanceada)
         │
         ▼
    generar_respuesta  ← Gemini + RAG context + historial
         │
         ▼
        END
```

## 1. Instalación y configuración

In [ ]:
# %pip install langchain langchain-google-genai langchain-community langchain-text-splitters langchain-experimental langgraph chromadb pypdf pdfplumber python-dotenv pandas sentence-transformers

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEYS = [
    os.getenv("GOOGLE_API_KEY"),
    os.getenv("GOOGLE_API_KEY_2"),
    os.getenv("GOOGLE_API_KEY_3"),
]
GOOGLE_API_KEYS = [k for k in GOOGLE_API_KEYS if k]  # elimina vacías/None

assert GOOGLE_API_KEYS, "No hay ninguna GOOGLE_API_KEY configurada en el archivo .env"
print(f"Claves API cargadas: {len(GOOGLE_API_KEYS)}")

## 2. Carga y procesado de documentos

In [ ]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
import pdfplumber

# Rutas organizadas por tipo
PRACTICOS_ES = Path("../data/manuales/practicos/es")
WEB_ES       = Path("../data/manuales/web/es")

MANUAL_METADATA = {
    (PRACTICOS_ES, "manual_iva_303_2025.pdf"):                   {"modelos": "303",     "perfil": "ambos",    "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_actividades_economicas_111_115.pdf"):  {"modelos": "111,115", "perfil": "ambos",    "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_renta_100_130_2025_parte1.pdf"):       {"modelos": "100,130", "perfil": "autonomo", "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_renta_100_130_2025_parte2.pdf"):       {"modelos": "100,130", "perfil": "autonomo", "tipo": "manual_practico", "idioma": "es"},
    (PRACTICOS_ES, "manual_sociedades_200_202_2024.pdf"):         {"modelos": "200,202", "perfil": "sociedad", "tipo": "manual_practico", "idioma": "es"},
    (WEB_ES, "manual_rentaweb_100_2024.pdf"):                     {"modelos": "100",     "perfil": "autonomo", "tipo": "manual_web",      "idioma": "es"},
    (WEB_ES, "manual_sociedadesweb_200_2024.pdf"):                {"modelos": "200",     "perfil": "sociedad", "tipo": "manual_web",      "idioma": "es"},
}

CHROMA_DIR_CHECK = Path("../chroma_db")

if CHROMA_DIR_CHECK.exists():
    docs_manuales = []
    print("Base de conocimiento ya indexada — carga de PDFs omitida.")
else:
    # SemanticChunker respeta fronteras conceptuales (artículos, apartados) en lugar de
    # cortar por número fijo de caracteres. Crítico para documentos fiscales donde un
    # apartado completo es la unidad mínima de información coherente.
    # Se inicializa aquí para reutilizar los embeddings ya cargados más abajo en la sección 3,
    # pero como la celda de embeddings aún no ejecutó, usamos un bloque try/except que
    # permite ejecutar las celdas en orden o reutilizar embeddings si ya existen.
    try:
        _embeddings_chunker = embeddings  # reutiliza si la sección 3 ya ejecutó
    except NameError:
        from langchain_community.embeddings import HuggingFaceEmbeddings
        _embeddings_chunker = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
        )

    semantic_splitter = SemanticChunker(
        embeddings=_embeddings_chunker,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=95,
    )
    # Fallback para textos muy cortos donde el chunker semántico no puede actuar
    fallback_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)

    def cargar_pdfs(metadata_map: dict) -> list:
        """Carga PDFs con pdfplumber, divide semánticamente y añade metadatos."""
        docs = []
        for (directorio, filename), meta in metadata_map.items():
            path = directorio / filename
            if not path.exists():
                print(f"  [AVISO] No encontrado: {path}")
                continue
            texto_completo = []
            with pdfplumber.open(str(path)) as pdf:
                for page in pdf.pages:
                    texto = page.extract_text()
                    if texto:
                        texto_completo.append(texto)
            texto_unido = "\n\n".join(texto_completo)
            doc_base = Document(page_content=texto_unido, metadata={**meta, "fuente": filename})
            try:
                chunks = semantic_splitter.split_documents([doc_base])
                if not chunks:
                    raise ValueError("SemanticChunker devolvió 0 chunks")
            except Exception:
                chunks = fallback_splitter.split_documents([doc_base])
            docs.extend(chunks)
            print(f"  [{meta['idioma']}] {filename}: {len(chunks)} chunks (semántico)")
        return docs

    print("Cargando manuales con chunking semántico...")
    docs_manuales = cargar_pdfs(MANUAL_METADATA)
    print(f"\nTotal chunks manuales: {len(docs_manuales)}")

In [ ]:
import pandas as pd
from langchain_core.documents import Document

CALENDARIO_PATH   = Path("../data/calendario_fiscal.csv")
OBLIGACIONES_PATH = Path("../data/obligaciones_perfil.csv")

if CHROMA_DIR_CHECK.exists():
    docs_calendario   = []
    docs_obligaciones = []
    print("CSVs omitidos — base de conocimiento ya indexada.")
else:
    def cargar_csv_como_docs(path: Path, tipo: str, sep: str = ",") -> list:
        df = pd.read_csv(path, sep=sep)
        docs = []
        for _, row in df.iterrows():
            contenido = " | ".join(f"{col}: {val}" for col, val in row.items() if pd.notna(val))
            meta = {"fuente": path.name, "tipo": tipo}
            if "modelo"   in row: meta["modelos"]   = str(row["modelo"])
            if "perfil"   in row: meta["perfil"]    = str(row["perfil"])
            if "trimestre" in row: meta["trimestre"] = str(row["trimestre"])
            docs.append(Document(page_content=contenido, metadata=meta))
        return docs

    docs_calendario   = cargar_csv_como_docs(CALENDARIO_PATH,   "calendario",         sep=",")
    docs_obligaciones = cargar_csv_como_docs(OBLIGACIONES_PATH, "obligaciones_perfil", sep=";")

    print(f"Chunks calendario:   {len(docs_calendario)}")
    print(f"Chunks obligaciones: {len(docs_obligaciones)}")
    print(f"\nTotal a indexar: {len(docs_manuales) + len(docs_calendario) + len(docs_obligaciones)}")


## 3. Creación de la base de conocimiento vectorial

In [ ]:
import chromadb
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

CHROMA_DIR = "../chroma_db"
COLLECTION_NAME = "base_fiscal"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

if Path(CHROMA_DIR).exists():
    vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=embeddings,
        collection_name=COLLECTION_NAME
    )
    print(f"Base de conocimiento cargada desde disco: {vectorstore._collection.count()} documentos")
else:
    all_docs = docs_manuales + docs_calendario + docs_obligaciones
    print(f"Total documentos a indexar: {len(all_docs)}")
    print("Indexando en local... (puede tardar 5-10 min)")

    vectorstore = Chroma.from_documents(
        documents=all_docs,
        embedding=embeddings,
        persist_directory=CHROMA_DIR,
        collection_name=COLLECTION_NAME
    )
    print(f"\nBase de conocimiento creada: {vectorstore._collection.count()} documentos")


In [ ]:
# Verificar la colección con consultas de prueba
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

test_queries = [
    "¿Cuándo es el plazo del modelo 303 del primer trimestre?",
    "¿Cómo se calcula la base imponible del IVA?",
    "¿Qué obligaciones fiscales tiene un autónomo en el primer trimestre?",
]

for q in test_queries:
    print(f"\nConsulta: {q}")
    results = retriever.invoke(q)
    for r in results:
        print(f"  [{r.metadata.get('fuente', '?')}] {r.page_content[:120]}...")

# Diagnóstico: ver qué idioma tiene el texto extraído de los manuales de renta
print("\n\n--- DIAGNÓSTICO: primeras líneas del manual de renta ---")
if docs_manuales:
    for doc in docs_manuales[:3]:
        if "renta" in doc.metadata.get("fuente", ""):
            print(f"Fuente: {doc.metadata['fuente']}")
            print(f"Texto: {doc.page_content[:300]}")
            print("---")


Consulta: ¿Cuándo es el plazo del modelo 303 del primer trimestre?
  [manual_renta_100_130_2025_parte1.pdf] a. Periodo inicial: del 01-07-25 al 31-12-25 (184 días)
Es el periodo comprendido entre el día siguiente a la fecha de v...
  [manual_sociedades_200_202_2024.pdf] presentació de la declaració haurà de realitzar-se durant els vint primers dies naturals dels 
mesos d'abril, juliol, oc...
  [manual_renta_100_130_2025_parte1.pdf] b. Període final :  del 01-01-26 al 31-06-26 (181 dies)
És el període comprès entre el dia 1 de gener de 2026 i el dia d...
  [manual_renta_100_130_2025_parte1.pdf] 2.2 Interessos de demora corresponents a la deducció indeguda de 2023 
(50.000 euros) 
a. Període inicial : del 02-07-24...

Consulta: ¿Cómo se calcula la base imponible del IVA?
  [manual_actividades_economicas_111_115.pdf] 5.3.2 En qué consiste
Quien realice entregas de bienes o prestaciones de servicios repercutirá el tipo impositivo 
del I...
  [manual_actividades_economicas_111_115.pdf] 5.3

## 4. Diseño del agente LangGraph

In [ ]:
SYSTEM_PROMPT = """Eres un asesor fiscal experto de una gestoría española llamada GestorIA.
Tu función es ayudar a gestores y clientes con las obligaciones fiscales de autónomos y sociedades en España.

## ROL Y LÍMITES

Eres un asistente especializado EXCLUSIVAMENTE en fiscalidad española. No respondas preguntas fuera de este ámbito.
Si te preguntan algo que no es fiscal (contabilidad general, derecho laboral, etc.), indica amablemente que está fuera de tu alcance.

## IDIOMA

Detecta el idioma en que escribe el usuario y responde siempre en ese mismo idioma.
El idioma de los documentos recuperados (contexto) NO influye en tu idioma de respuesta.

## FUENTES Y JERARQUÍA

Aplica siempre esta prioridad al responder:

1. **Contexto RAG** (documentos recuperados) — máxima autoridad para datos concretos: fechas, casillas, porcentajes, plazos, importes. Si el RAG y tu conocimiento general difieren, prevalece siempre el RAG.
2. **Conocimiento general como asesor fiscal** — solo para procedimientos estándar (acceso a sede AEAT, Cl@ve, certificado digital). Cita como "Procedimiento estándar AEAT".
3. **Ninguna fuente** — si no hay datos en ninguna de las dos fuentes anteriores, declara la ausencia explícitamente.

Si tienes información parcial, responde con lo que tengas y señala qué falta: "Sobre X dispongo de [dato], pero no tengo información sobre Y en mi base de conocimiento."
Solo cierra con "No dispongo de información suficiente..." cuando no tengas absolutamente ningún dato relevante.

## FUENTES — CÓMO CITARLAS

Al citar la fuente usa el nombre descriptivo, no solo el nombre de fichero:
- `calendario_fiscal.csv` → "Calendario fiscal AEAT 2026"
- `obligaciones_perfil.csv` → "Mapa de obligaciones por perfil"
- `manual_iva_303_2025.pdf` → "Manual práctico IVA 303 (AEAT 2025)"
- `manual_renta_100_130_2025_parte1.pdf` / `parte2.pdf` → "Manual práctico Renta 100/130 (AEAT 2025)"
- `manual_sociedades_200_202_2024.pdf` → "Manual práctico Sociedades 200/202 (AEAT 2024)"
- `manual_actividades_economicas_111_115.pdf` → "Manual Actividades Económicas 111/115 (AEAT)"
- `manual_rentaweb_100_2024.pdf` → "Manual RentaWeb 100 (AEAT 2024)"
- `manual_sociedadesweb_200_2024.pdf` → "Manual SociedadesWeb 200 (AEAT 2024)"
- Conocimiento propio de procedimiento → "Procedimiento estándar AEAT"

## VIGENCIA DEL CONTEXTO

Si en los fragmentos RAG recuperados detectas referencias a ejercicios anteriores al trimestre actual (por ejemplo, menciones a "2023", "2024" o años anteriores en fechas de plazo o nombres de modelos), añade al final de tu respuesta: "⚠️ Parte del contexto recuperado puede corresponder a ejercicios anteriores. Verifica los datos en la sede electrónica de la AEAT (sede.agenciatributaria.gob.es) antes de actuar."
No añadas este aviso si los fragmentos son coherentes con el ejercicio fiscal actual (2025–2026).

## IDENTIFICACIÓN DE PERFIL

- Identifica el perfil del cliente antes de responder: autónomo, sociedad, o ambos.
- Si el perfil aparece en la línea "Perfil del cliente" al inicio del mensaje, úsalo directamente sin volver a preguntar.
- Si el perfil NO está claro ni en esa línea ni en el historial, PREGUNTA antes de responder. No asumas.
- Una vez identificado, el perfil persiste durante toda la conversación. Inclúyelo en la primera respuesta y en aquellas donde aporte claridad (cambio de modelo, respuesta larga). En respuestas de seguimiento cortas puede omitirse si ya es evidente del contexto.
- En preguntas de seguimiento ("¿y el 130?", "¿cuánto tengo que pagar?"), usa el perfil y contexto del turno anterior sin solicitar aclaración si la pregunta es razonablemente interpretable.
- Si el modelo preguntado no aplica al perfil del cliente, indícalo antes de responder y redirige al modelo correcto cuando sea posible. Ejemplos: "El modelo 130 no aplica a sociedades — el equivalente es el modelo 202." / "El modelo 200 es exclusivo de sociedades — los autónomos liquidan el IRPF con el modelo 100."

## NIVEL TÉCNICO

Adapta el nivel de detalle según quién pregunta:
- **Gestor / asesor fiscal** — usa terminología técnica (casillas, regímenes, base imponible, devengo). Respuestas densas y precisas.
- **Cliente final** — lenguaje claro, sin jerga. Explica brevemente qué significa cada término técnico la primera vez que lo uses.

Si no está claro el tipo de interlocutor, usa un nivel intermedio: terminología técnica con una frase de contexto cuando sea necesario.

## ESTRUCTURA DE RESPUESTA

Adapta la estructura al tipo de pregunta. Incluye SOLO los apartados relevantes:

**Preguntas de plazo o calendario** → Perfil | Modelo + fecha límite + domiciliación + inicio preparación | Fuente
**Preguntas de cumplimentación o casillas** → Perfil | Nombre de la casilla + explicación técnica | Fuente
**Preguntas de procedimiento o pasos** → Perfil | Lista numerada de pasos | Fuente
**Preguntas de obligaciones generales** → Perfil | Lista de modelos aplicables con plazo e inicio preparación | Fuente
**Preguntas mixtas** → combina las secciones necesarias en orden lógico, sin duplicar información.

Nunca incluyas secciones vacías ni encabezados sin contenido. Si la pregunta solo pide un dato concreto (una fecha, una casilla), responde directamente.

## PLAZOS Y ANTELACIÓN

- La fecha de hoy y el trimestre activo aparecen en la línea "Fecha de hoy — Trimestre actual" del mensaje. Úsalos para resolver preguntas sin trimestre explícito ("¿qué tengo pendiente?", "¿el trimestre que viene?"). Si esa línea no está presente, deduce el trimestre a partir de tu conocimiento de la fecha actual: enero–marzo=1T, abril–junio=2T, julio–septiembre=3T, octubre–diciembre=4T.
- Días de preparación recomendados por tipo de obligación (úsalos si el contexto RAG no especifica otro valor):
  - Modelos trimestrales (303, 130, 111, 115, 202): 10 días antes del plazo
  - Modelos anuales simples (390, 347): 15 días antes del plazo
  - Modelos anuales complejos (100, 200): 30 días antes del plazo
- Cuando informes de un plazo, calcula y muestra siempre la fecha de inicio de preparación.
- Si el usuario pregunta "¿qué tengo pendiente?", lista TODAS las obligaciones del trimestre activo ordenadas por fecha límite.

## CORRECCIÓN DE ERRORES DEL USUARIO

Si detectas una incoherencia en la pregunta (trimestre incorrecto para ese modelo, fecha imposible, modelo que no aplica al perfil), corrígela de forma breve y directa antes de responder:
"El modelo 130 no tiene presentación en el cuarto trimestre — el último es en octubre (3T). Te respondo sobre el 3T:"

## CONTEXTO ACUMULADO EN CONVERSACIÓN

Si el usuario hace varias preguntas encadenadas sobre el mismo modelo o tema, no repitas explicaciones ya dadas en el mismo hilo. Céntrate solo en la información nueva que aporta la pregunta actual. Si necesitas referirte a algo ya explicado, usa una referencia breve: "Como comenté antes, el plazo es el 20 de julio."

## TONO

Profesional y directo. Usa listas y negritas para facilitar la lectura.
Evita relleno vacío ("¡Claro!", "¡Por supuesto!") pero puedes usar una transición breve cuando el contexto lo pida ("En ese caso," "Para este perfil,").

---

## EJEMPLOS DE RESPUESTA CORRECTA

**Ejemplo 1 — Plazo de un modelo concreto:**
Usuario: "Soy autónomo, ¿cuándo presento el 303 del 2T?"

Respuesta:
**Perfil:** Autónomo.
**Modelo 303 — Autoliquidación IVA 2T 2026:**
- Fecha límite: 20 de julio de 2026
- Domiciliación hasta: 15 de julio de 2026
- Inicio de preparación recomendado: 10 de julio de 2026 (10 días antes)
*Fuente: Calendario fiscal AEAT 2026*

---

**Ejemplo 2 — Perfil no especificado:**
Usuario: "¿Cuándo tengo que presentar el modelo 303?"

Respuesta:
Para darte la información correcta, necesito saber tu perfil fiscal. ¿Eres autónomo o representas a una sociedad?

---

**Ejemplo 3 — Cómo rellenar una casilla:**
Usuario: "Soy autónomo. ¿Cómo relleno la casilla 01 del modelo 303?"

Respuesta:
**Perfil:** Autónomo.
**Casilla 01 — Base imponible al tipo general (21%):**
Incluye el importe total de las entregas de bienes y prestaciones de servicios sujetas y no exentas de IVA gravadas al 21%, sin incluir el propio impuesto.
*Fuente: Manual práctico IVA 303 (AEAT 2025)*

---

**Ejemplo 4 — Obligaciones de un trimestre:**
Usuario: "Somos una S.L. ¿Qué tenemos que presentar en el segundo trimestre?"

Respuesta:
**Perfil:** Sociedad.
**Obligaciones 2T 2026:**
- Modelo 303 — IVA 2T | Plazo: 20 julio | Inicio preparación: 10 julio
- Modelo 111 — Retenciones IRPF 2T | Plazo: 20 julio | Inicio preparación: 10 julio
- Modelo 115 — Retenciones alquileres 2T | Plazo: 20 julio | Inicio preparación: 10 julio
- Modelo 202 — Pago fraccionado IS | Plazo: 20 julio | Inicio preparación: 10 julio
*Fuente: Calendario fiscal AEAT 2026, Mapa de obligaciones por perfil*

---

**Ejemplo 5 — Información no disponible:**
Usuario: "¿Cuál es el tipo de recargo de equivalencia para el aceite de oliva?"

Respuesta:
No dispongo de información suficiente sobre este punto en mi base de conocimiento. Te recomiendo consultar la sede electrónica de la AEAT (sede.agenciatributaria.gob.es) o al gestor responsable.

---

**Ejemplo 6 — Pasos para presentar un modelo:**
Usuario: "Represento una sociedad. ¿Qué pasos debo seguir para presentar el IVA?"

Respuesta:
**Perfil:** Sociedad.
**Pasos para presentar el Modelo 303 — Autoliquidación IVA:**

1. **Recopilar la documentación** — Facturas emitidas (IVA repercutido) y facturas recibidas (IVA soportado) del trimestre.
2. **Calcular el resultado** — IVA repercutido − IVA soportado deducible. Si es positivo, hay que ingresar; si es negativo, se puede compensar o solicitar devolución.
3. **Acceder a la Sede Electrónica de la AEAT** — Con certificado digital, DNIe o Cl@ve PIN en sede.agenciatributaria.gob.es.
4. **Cumplimentar el formulario** — Ir a "Trámites destacados > Modelo 303". Rellenar casillas 01-09 (IVA devengado) y 28-44 (IVA deducible).
5. **Presentar y pagar si procede** — Se puede domiciliar hasta 5 días antes del plazo límite.
6. **Guardar el justificante** — La AEAT emite un CSV de confirmación como acuse de recibo oficial.

*Fuente: Manual práctico IVA 303 (AEAT 2025), procedimiento estándar AEAT*

---

**Ejemplo 7 — Pregunta de seguimiento (perfil ya conocido):**
[Turno anterior: el usuario dijo ser autónomo y preguntó por el modelo 303]
Usuario: "¿Y el 130?"

Respuesta:
**Modelo 130 — Pago fraccionado IRPF 2T 2026** (Autónomo, estimación directa):
- Fecha límite: 20 de julio de 2026
- Inicio de preparación recomendado: 10 de julio de 2026
*Fuente: Calendario fiscal AEAT 2026*

---

**Ejemplo 8 — Información parcialmente disponible:**
Usuario: "Soy autónomo. ¿Cuánto tengo que ingresar en el modelo 130 y cuál es la penalización por presentarlo tarde?"

Respuesta:
**Perfil:** Autónomo.
**Modelo 130 — Cálculo del importe:** El importe es el 20% del rendimiento neto de actividades económicas del trimestre, menos retenciones soportadas y pagos fraccionados anteriores del ejercicio.
*Fuente: Manual práctico Renta 100/130 (AEAT 2025)*

Sobre las penalizaciones por presentación fuera de plazo no dispongo de información en mi base de conocimiento. Te recomiendo consultar la sede electrónica de la AEAT (sede.agenciatributaria.gob.es) para conocer el régimen de recargos aplicable.

---

**Ejemplo 9 — Corrección de error del usuario:**
Usuario: "Soy autónomo, ¿cuándo presento el modelo 130 del cuarto trimestre?"

Respuesta:
El modelo 130 no tiene presentación en el cuarto trimestre — el último pago fraccionado del año es el 3T (octubre). En enero se presenta el modelo 100 con la liquidación anual del IRPF.

**Modelo 130 — Pago fraccionado IRPF 3T 2026** (Autónomo):
- Fecha límite: 20 de octubre de 2026
- Inicio de preparación recomendado: 10 de octubre de 2026
*Fuente: Calendario fiscal AEAT 2026*

---

**Ejemplo 10 — Pregunta mixta (plazo + procedimiento):**
Usuario: "Soy autónomo. ¿Cuándo tengo que presentar el 303 del 1T y cómo lo hago?"

Respuesta:
**Perfil:** Autónomo.

**Plazo — Modelo 303 IVA 1T 2026:**
- Fecha límite: 20 de abril de 2026
- Domiciliación hasta: 15 de abril de 2026
- Inicio de preparación recomendado: 10 de abril de 2026
*Fuente: Calendario fiscal AEAT 2026*

**Cómo presentarlo:**
1. Accede a sede.agenciatributaria.gob.es con certificado digital o Cl@ve PIN.
2. Ve a "Trámites destacados > Modelo 303" y cumplimenta las casillas con los datos del trimestre.
3. Presenta y, si sale a ingresar, puedes domiciliar el pago hasta el 15 de abril.
4. Guarda el justificante con el CSV de confirmación.
*Fuente: procedimiento estándar AEAT*

---

**Ejemplo 11 — Modelo no aplicable al perfil:**
Usuario: "Soy autónomo, ¿cuándo presento el modelo 202?"

Respuesta:
El modelo 202 (Pagos fraccionados del Impuesto sobre Sociedades) no aplica a autónomos — es exclusivo de sociedades. Como autónomo, el pago fraccionado del IRPF se realiza con el **modelo 130** (estimación directa) o el **modelo 131** (estimación objetiva/módulos).

**Modelo 130 — Pago fraccionado IRPF 2T 2026** (Autónomo):
- Fecha límite: 20 de julio de 2026
- Inicio de preparación recomendado: 10 de julio de 2026
*Fuente: Calendario fiscal AEAT 2026, Mapa de obligaciones por perfil*
"""

print("System prompt configurado con 11 few-shot examples.")

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, RemoveMessage
import operator
import re
import datetime
import time

# ── Constantes ──────────────────────────────────────────────────────────────
MAX_MESSAGES    = 10
MAX_RETRIES_RPM = 3

class AgentState(TypedDict):
    messages:     Annotated[list, operator.add]
    perfil:       str
    contexto_rag: str
    tipo_consulta: str


def _extraer_retry_delay(error_str: str, default: float = 15.0) -> float:
    match = re.search(r"retryDelay.*?(\d+(?:\.\d+)?)\s*s", error_str)
    return float(match.group(1)) + 1 if match else default


def _es_limite_diario(err: str) -> bool:
    return "GenerateRequestsPerDayPerProjectPerModel" in err

def _es_limite_rpm(err: str) -> bool:
    return "GenerateRequestsPerMinutePerProjectPerModel" in err


# Estado global del índice de clave activa (permite rotar en caliente)
_estado_claves = {"idx": 0}

def _crear_llm_para_clave(idx: int) -> ChatGoogleGenerativeAI:
    return ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GOOGLE_API_KEYS[idx],
        temperature=0,
    )

def _inicializar_llm(claves: list) -> ChatGoogleGenerativeAI:
    """Prueba las claves en orden al arrancar y devuelve el LLM con la primera disponible."""
    if not claves:
        raise RuntimeError("No hay ninguna GOOGLE_API_KEY configurada.")
    for i in range(len(claves)):
        for intento in range(MAX_RETRIES_RPM):
            try:
                llm_test = _crear_llm_para_clave(i)
                llm_test.invoke([HumanMessage(content="ok")])
                if i > 0:
                    print(f"[INFO] Arrancando con clave API {i + 1}.")
                _estado_claves["idx"] = i
                return llm_test
            except Exception as e:
                err = str(e)
                if "RESOURCE_EXHAUSTED" not in err and "429" not in err:
                    raise
                if _es_limite_diario(err):
                    print(f"[WARN] Clave {i + 1} agotada (límite diario), probando siguiente...")
                    break
                delay = _extraer_retry_delay(err)
                print(f"[WARN] Clave {i + 1} — límite RPM al arrancar, esperando {delay:.0f}s...")
                time.sleep(delay)
        else:
            continue
        continue
    raise RuntimeError("Todas las claves de API están agotadas al arrancar.")


def _invoke_con_retry(llm_obj, messages: list, max_retries: int = MAX_RETRIES_RPM):
    """Invoca el LLM con retry RPM y fallback a siguiente clave si hay límite diario."""
    global llm, llm_con_tools

    for intento in range(max_retries + len(GOOGLE_API_KEYS)):
        try:
            return llm_obj.invoke(messages)
        except Exception as e:
            err = str(e)
            if "RESOURCE_EXHAUSTED" not in err and "429" not in err:
                raise

            if _es_limite_rpm(err):
                delay = _extraer_retry_delay(err)
                print(f"[WARN] Límite RPM — esperando {delay:.0f}s (intento {intento + 1})...")
                time.sleep(delay)
                continue

            if _es_limite_diario(err):
                idx_actual = _estado_claves["idx"]
                idx_nuevo  = idx_actual + 1
                if idx_nuevo >= len(GOOGLE_API_KEYS):
                    raise RuntimeError("Todas las claves de API están agotadas (límite diario).") from e
                print(f"[WARN] Clave {idx_actual + 1} agotada (límite diario) — cambiando a clave {idx_nuevo + 1}...")
                _estado_claves["idx"] = idx_nuevo
                nueva_llm = _crear_llm_para_clave(idx_nuevo)
                # Actualizar las referencias globales para que el resto del notebook use la nueva clave
                llm           = nueva_llm
                llm_con_tools = nueva_llm.bind_tools(tools_fiscales) if "tools_fiscales" in dir() else nueva_llm
                llm_obj       = nueva_llm if llm_obj is not nueva_llm else llm_obj
                continue

            raise

    raise RuntimeError("Se agotaron los reintentos de la API.")


llm = _inicializar_llm(GOOGLE_API_KEYS)

# ── Gestión de tokens: poda de historial ────────────────────────────────────
def podar_historial(state: AgentState) -> AgentState:
    mensajes = state["messages"]
    if len(mensajes) <= MAX_MESSAGES:
        return {}
    n_eliminar = len(mensajes) - MAX_MESSAGES
    if n_eliminar % 2 != 0:
        n_eliminar += 1
    ids_a_eliminar = [RemoveMessage(id=m.id) for m in mensajes[:n_eliminar]]
    print(f"[Historial] Podados {n_eliminar} mensajes antiguos. Quedan {len(mensajes) - n_eliminar}.")
    return {"messages": ids_a_eliminar}

# ── Clasificador: detecta tipo de consulta ──────────────────────────────────
_KEYWORDS_PLAZOS = re.compile(
    r"\b(plazo|fecha|cuando|cuándo|vencimiento|trimestre|domicili|antelacion|antelación|pendiente)\b",
    re.IGNORECASE
)
_KEYWORDS_DOCS = re.compile(
    r"\b(casilla|rellenar|cumplimentar|calcul|base imponible|deduccion|deducción|como se|cómo se|"
    r"instruccion|instrucción|apartado|anexo|paso|pasos|proceso|procedimiento|"
    r"c[oó]mo presento|c[oó]mo se presenta|c[oó]mo funciona|c[oó]mo hago|"
    r"c[oó]mo debo|qué pasos|qu[eé] debo hacer)\b",
    re.IGNORECASE
)

def clasificar_consulta(state: AgentState) -> AgentState:
    ultima = state["messages"][-1].content
    if _KEYWORDS_DOCS.search(ultima):
        tipo = "documentos"
    elif _KEYWORDS_PLAZOS.search(ultima):
        tipo = "plazos"
    else:
        tipo = "general"
    print(f"[Router] Consulta clasificada como: '{tipo}'")
    return {"tipo_consulta": tipo}

def router(state: AgentState) -> Literal["recuperar_plazos", "recuperar_documentos", "recuperar_general"]:
    return {
        "plazos":     "recuperar_plazos",
        "documentos": "recuperar_documentos",
        "general":    "recuperar_general",
    }[state["tipo_consulta"]]

# ── Nodos de recuperación RAG ────────────────────────────────────────────────
def _combinar_docs(docs_a: list, docs_b: list) -> str:
    vistos = set()
    combinados = []
    for doc in docs_a + docs_b:
        clave = doc.page_content[:100]
        if clave not in vistos:
            vistos.add(clave)
            combinados.append(doc)
    resultado = "\n\n".join(
        f"[{d.metadata.get('fuente', '?')}]\n{d.page_content}"
        for d in combinados
    )
    print(f"[RAG] Documentos recuperados: {len(combinados)} "
          f"({', '.join(set(d.metadata.get('fuente','?') for d in combinados))})")
    return resultado

def recuperar_plazos(state: AgentState) -> AgentState:
    ultima = state["messages"][-1].content
    perfil = state.get("perfil", "")
    retriever_csv = vectorstore.as_retriever(search_kwargs={"k": 10, "filter": {"tipo": {"$in": ["calendario", "obligaciones_perfil"]}}})
    docs_csv = retriever_csv.invoke(ultima)
    search_manuales = {"k": 3}
    if perfil in ("autonomo", "sociedad"):
        search_manuales["filter"] = {"perfil": {"$in": [perfil, "ambos"]}}
    docs_manuales_res = vectorstore.as_retriever(search_kwargs=search_manuales).invoke(ultima)
    return {"contexto_rag": _combinar_docs(docs_csv, docs_manuales_res)}

def recuperar_documentos(state: AgentState) -> AgentState:
    ultima = state["messages"][-1].content
    perfil = state.get("perfil", "")
    search_manuales = {"k": 8}
    if perfil in ("autonomo", "sociedad"):
        search_manuales["filter"] = {"perfil": {"$in": [perfil, "ambos"]}}
    docs_manuales_res = vectorstore.as_retriever(search_kwargs=search_manuales).invoke(ultima)
    retriever_csv = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"tipo": {"$in": ["calendario", "obligaciones_perfil"]}}})
    docs_csv = retriever_csv.invoke(ultima)
    return {"contexto_rag": _combinar_docs(docs_manuales_res, docs_csv)}

def recuperar_general(state: AgentState) -> AgentState:
    ultima = state["messages"][-1].content
    perfil = state.get("perfil", "")
    search_kwargs_manuales = {"k": 5}
    if perfil in ("autonomo", "sociedad"):
        search_kwargs_manuales["filter"] = {"perfil": {"$in": [perfil, "ambos"]}}
    docs_manuales_res = vectorstore.as_retriever(search_kwargs=search_kwargs_manuales).invoke(ultima)
    retriever_csv = vectorstore.as_retriever(search_kwargs={"k": 6, "filter": {"tipo": {"$in": ["calendario", "obligaciones_perfil"]}}})
    docs_csv = retriever_csv.invoke(ultima)
    return {"contexto_rag": _combinar_docs(docs_manuales_res, docs_csv)}

# ── Nodo de generación ───────────────────────────────────────────────────────
_KW_AUTO = re.compile(r"\baut[oó]nomo\b", re.IGNORECASE)
_KW_SOC  = re.compile(r"\b(sociedad|empresa|s\.l|s\.a)\b", re.IGNORECASE)

def generar_respuesta(state: AgentState) -> AgentState:
    contexto  = state.get("contexto_rag", "")
    historial = state["messages"]

    hoy = datetime.date.today()
    trimestre = (hoy.month - 1) // 3 + 1
    contexto_temporal = f"Fecha de hoy: {hoy.strftime('%d/%m/%Y')} — Trimestre actual: {trimestre}T 2026\n"

    perfil_actual = state.get("perfil", "")
    perfil_linea  = ""
    if perfil_actual:
        label = {"autonomo": "Autónomo", "sociedad": "Sociedad"}.get(perfil_actual, "")
        perfil_linea = f"Perfil del cliente: {label}\n"

    messages = [SystemMessage(content=SYSTEM_PROMPT)]
    messages += historial[:-1]
    prompt_con_contexto = (
        f"{contexto_temporal}{perfil_linea}"
        f"Contexto recuperado de la base de conocimiento:\n---\n{contexto}\n---\n\n"
        f"Pregunta del usuario: {historial[-1].content}"
    )
    messages.append(HumanMessage(content=prompt_con_contexto))
    respuesta = _invoke_con_retry(llm, messages)

    if not perfil_actual:
        for msg in reversed(historial):
            texto = msg.content
            if _KW_AUTO.search(texto):
                perfil_actual = "autonomo"
                break
            if _KW_SOC.search(texto):
                perfil_actual = "sociedad"
                break

    return {
        "messages": [AIMessage(content=respuesta.content)],
        "perfil": perfil_actual,
    }

# ── Construcción del grafo ───────────────────────────────────────────────────
workflow = StateGraph(AgentState)
workflow.add_node("podar_historial",      podar_historial)
workflow.add_node("clasificar_consulta",  clasificar_consulta)
workflow.add_node("recuperar_plazos",     recuperar_plazos)
workflow.add_node("recuperar_documentos", recuperar_documentos)
workflow.add_node("recuperar_general",    recuperar_general)
workflow.add_node("generar_respuesta",    generar_respuesta)
workflow.add_edge(START, "podar_historial")
workflow.add_edge("podar_historial", "clasificar_consulta")
workflow.add_conditional_edges("clasificar_consulta", router, {
    "recuperar_plazos":     "recuperar_plazos",
    "recuperar_documentos": "recuperar_documentos",
    "recuperar_general":    "recuperar_general",
})
workflow.add_edge("recuperar_plazos",     "generar_respuesta")
workflow.add_edge("recuperar_documentos", "generar_respuesta")
workflow.add_edge("recuperar_general",    "generar_respuesta")
workflow.add_edge("generar_respuesta",    END)

memory = MemorySaver()
agente = workflow.compile(checkpointer=memory)

print("Agente LangGraph compilado — clave activa:", _estado_claves["idx"] + 1)
print("  ✓ Retry RPM automático + fallback de clave ante límite diario")

## 5. Lógica de avisos por antelación

In [ ]:
from datetime import date, timedelta

def obtener_obligaciones_proximas(perfil: str, dias_horizonte: int = 60) -> str:
    """Devuelve las obligaciones fiscales próximas en los próximos N días."""
    df = pd.read_csv("../data/calendario_fiscal.csv", sep=",")
    hoy = date.today()
    limite = hoy + timedelta(days=dias_horizonte)

    if perfil in ("autonomo", "sociedad"):
        df = df[df["perfil"].isin([perfil, "ambos"])]

    df["fecha_limite_2026"] = pd.to_datetime(df["fecha_limite_2026"]).dt.date
    df = df[(df["fecha_limite_2026"] >= hoy) & (df["fecha_limite_2026"] <= limite)]
    df = df.sort_values("fecha_limite_2026")

    if df.empty:
        return f"No hay obligaciones fiscales en los próximos {dias_horizonte} días."

    lineas = [f"Obligaciones próximas ({hoy} → {limite}):\n"]
    for _, row in df.iterrows():
        fecha = row["fecha_limite_2026"]
        inicio = fecha - timedelta(days=int(row["dias_preparacion_recomendados"]))
        lineas.append(
            f"• Modelo {row['modelo']} — {row['nombre']}\n"
            f"  Plazo: {fecha} | Inicio recomendado: {inicio}\n"
        )
    return "\n".join(lineas)

# Ejemplo
print(obtener_obligaciones_proximas("autonomo", dias_horizonte=90))

Obligaciones próximas (2026-05-03 → 2026-08-01):

• Modelo 100 — IRPF anual 2025 — fin campaña
  Plazo: 2026-06-30 | Inicio recomendado: 2026-05-31

• Modelo 130 — Pago fraccionado IRPF autónomos 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-10

• Modelo 303 — Autoliquidación IVA 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-10

• Modelo 111 — Retenciones e ingresos a cuenta IRPF 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-15

• Modelo 115 — Retenciones e ingresos a cuenta — alquileres 2T 2026
  Plazo: 2026-07-20 | Inicio recomendado: 2026-07-15



## 6. Moderación en cascada

Filtra preguntas fuera del ámbito fiscal **antes** de llegar al agente mediante tres capas progresivas:

1. **Reglas** — keywords fiscales/offtopic, coste cero, instantáneo
2. **ML** — TF-IDF + LogisticRegression, solo actúa si la confianza ≥ 85 %
3. **LLM** — llamada a Gemini únicamente para los casos ambiguos que superan las dos capas anteriores

Este patrón reduce las llamadas innecesarias a la API en ~80 % para preguntas claramente fuera de ámbito.

In [ ]:
import re
import numpy as np
from dataclasses import dataclass
from typing import Optional
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

UMBRAL_CONFIANZA_ML = 0.85

KEYWORDS_FISCAL = re.compile(
    r"\b(modelo|irpf|iva|impuesto|declaraci[oó]n|renta|hacienda|aeat|tribut|fiscal|"
    r"autonomo|aut[oó]nomo|sociedad|empresa|s\.l|factura|casilla|plazo|trimestre|"
    r"303|130|111|115|100|200|202|347|390|retenci[oó]n|deducci[oó]n)\b",
    re.IGNORECASE,
)
KEYWORDS_OFFTOPIC = re.compile(
    r"\b(receta|cocina|deporte|f[uú]tbol|pel[ií]cula|m[uú]sica|viaje|hotel|"
    r"tiempo|clima|meteorolog[ií]a|amor|relaci[oó]n|juego|videojuego)\b",
    re.IGNORECASE,
)

@dataclass
class ResultadoModeracion:
    decision:  str    # "fiscal" | "offtopic"
    capa:      str    # "reglas" | "ml" | "llm"
    confianza: float

_ejemplos = [
    ("¿Cuándo presento el modelo 303?",                        "fiscal"),
    ("¿Qué obligaciones tengo como autónomo?",                 "fiscal"),
    ("¿Cómo se rellena la casilla 01 del IVA?",               "fiscal"),
    ("Plazo para presentar el IRPF 2025",                      "fiscal"),
    ("¿Qué es la domiciliación en el modelo 130?",             "fiscal"),
    ("Retenciones en el modelo 111 del segundo trimestre",     "fiscal"),
    ("¿Cuánto tiempo tengo para presentar el IS?",             "fiscal"),
    ("Deducciones en el modelo 303",                           "fiscal"),
    ("¿Cómo me doy de alta como autónomo en hacienda?",        "fiscal"),
    ("¿Qué es el pago fraccionado del IRPF?",                  "fiscal"),
    ("¿Cuál es la mejor receta de paella?",                    "offtopic"),
    ("¿Quién ganó el partido de ayer?",                        "offtopic"),
    ("Recomiéndame una película de terror",                    "offtopic"),
    ("¿Qué tiempo hace en Madrid?",                            "offtopic"),
    ("¿Cómo se llama el presidente de Francia?",               "offtopic"),
    ("Cuéntame un chiste",                                     "offtopic"),
    ("¿Cuál es la capital de Australia?",                      "offtopic"),
    ("Dame una ruta de senderismo",                            "offtopic"),
]
_X, _y = zip(*_ejemplos)
clasificador_ml = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf",   LogisticRegression(max_iter=1000, random_state=42)),
])
clasificador_ml.fit(list(_X), list(_y))

def moderar_pregunta(texto: str) -> ResultadoModeracion:
    """Cascada: Reglas → ML → LLM para clasificar si la pregunta es fiscal."""
    if KEYWORDS_FISCAL.search(texto):
        return ResultadoModeracion("fiscal",   "reglas", 1.0)
    if KEYWORDS_OFFTOPIC.search(texto):
        return ResultadoModeracion("offtopic", "reglas", 1.0)

    proba     = clasificador_ml.predict_proba([texto])[0]
    clases    = clasificador_ml.classes_
    idx_max   = int(np.argmax(proba))
    confianza = float(proba[idx_max])
    if confianza >= UMBRAL_CONFIANZA_ML:
        return ResultadoModeracion(clases[idx_max], "ml", confianza)

    # Capa 3: LLM con retry completo
    prompt = (
        "Clasifica esta pregunta como 'fiscal' o 'offtopic'. "
        "Responde SOLO con una palabra.\n\nPregunta: " + texto
    )
    try:
        raw = _invoke_con_retry(llm, [HumanMessage(content=prompt)]).content.strip().lower()
        decision = "fiscal" if "fiscal" in raw else "offtopic"
        return ResultadoModeracion(decision, "llm", 0.6)
    except Exception as e:
        if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
            print(f"[WARN] Cuota API agotada en moderación — asumiendo 'fiscal' para: {texto[:60]}")
            return ResultadoModeracion("fiscal", "llm-fallback", 0.5)
        raise

# ── Prueba de la moderación ──────────────────────────────────────────────────
casos_prueba = [
    "¿Cuándo presento el modelo 303 del 2T?",
    "¿Cuál es la mejor receta de tortilla?",
    "Necesito información sobre retenciones IRPF",
    "¿Me recomiendas una serie en Netflix?",
    "¿Puedo deducir el alquiler de mi oficina?",
]

print("Prueba de moderación en cascada:\n")
for texto in casos_prueba:
    resultado = moderar_pregunta(texto)
    print(f"  [{resultado.decision.upper():8s}] [{resultado.capa:12s}] {texto}")

## 7. Herramientas fiscales con @tool y loop ReAct

En lugar de llamar siempre a las funciones del calendario, el agente decide **dinámicamente** cuándo invocarlas según la pregunta. El patrón ReAct crea un ciclo `agente → tools → agente` que termina cuando el LLM produce una respuesta final sin tool_calls.

Herramientas definidas:
- `consultar_obligaciones_proximas` — obligaciones en los próximos N días para un perfil
- `consultar_plazo_modelo` — plazo exacto de un modelo fiscal concreto

In [ ]:
import datetime
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

_df_cal = pd.read_csv("../data/calendario_fiscal.csv")
_df_cal["fecha_limite_2026"] = pd.to_datetime(_df_cal["fecha_limite_2026"]).dt.date

@tool
def consultar_obligaciones_proximas(perfil: str, dias_horizonte: int = 60) -> str:
    """Devuelve las obligaciones fiscales próximas para un perfil dado.
    Usar cuando el usuario pregunte qué tiene pendiente en los próximos días o meses.
    perfil: 'autonomo', 'sociedad' o 'ambos'
    dias_horizonte: número de días a mirar hacia adelante (por defecto 60)
    """
    hoy   = datetime.date.today()
    hasta = hoy + datetime.timedelta(days=dias_horizonte)
    df = _df_cal.copy()
    if perfil in ("autonomo", "sociedad"):
        df = df[df["perfil"].isin([perfil, "ambos"])]
    df = df[(df["fecha_limite_2026"] >= hoy) & (df["fecha_limite_2026"] <= hasta)]
    df = df.sort_values("fecha_limite_2026")
    if df.empty:
        return f"No hay obligaciones en los próximos {dias_horizonte} días."
    lineas = [f"Obligaciones {hoy} → {hasta}:\n"]
    for _, row in df.iterrows():
        inicio = row["fecha_limite_2026"] - datetime.timedelta(days=int(row["dias_preparacion_recomendados"]))
        lineas.append(
            f"• Modelo {row['modelo']} — {row['nombre']}\n"
            f"  Plazo: {row['fecha_limite_2026']} | Inicio recomendado: {inicio}"
        )
    return "\n".join(lineas)

@tool
def consultar_plazo_modelo(modelo: str) -> str:
    """Devuelve el plazo exacto de un modelo fiscal concreto.
    Usar cuando el usuario pregunte por la fecha límite de un modelo específico.
    modelo: número del modelo (ej: '303', '130', '111')
    """
    df = _df_cal[_df_cal["modelo"].astype(str) == str(modelo)]
    if df.empty:
        return f"No encontré información del modelo {modelo} en el calendario."
    lineas = []
    for _, row in df.iterrows():
        inicio = row["fecha_limite_2026"] - datetime.timedelta(days=int(row["dias_preparacion_recomendados"]))
        lineas.append(
            f"Modelo {row['modelo']} — {row['nombre']} ({row['perfil']})\n"
            f"  Plazo: {row['fecha_limite_2026']} | Inicio recomendado: {inicio}"
        )
    return "\n".join(lineas)

tools_fiscales = [consultar_obligaciones_proximas, consultar_plazo_modelo]
nodo_tools     = ToolNode(tools_fiscales)
llm_con_tools  = llm.bind_tools(tools_fiscales)

# ── Grafo ReAct con tools ────────────────────────────────────────────────────
from typing import Literal as _Literal

class AgentStateReAct(TypedDict):
    messages:     Annotated[list, operator.add]
    perfil:       str
    contexto_rag: str

def nodo_agente_react(state: AgentStateReAct) -> AgentStateReAct:
    contexto = state.get("contexto_rag", "")
    fecha_hoy = datetime.date.today().strftime("%d/%m/%Y")
    perfil_linea = ""
    if state.get("perfil"):
        label = {"autonomo": "Autónomo", "sociedad": "Sociedad"}.get(state["perfil"], "")
        perfil_linea = f"Perfil del cliente: {label}\n"
    system = (
        SYSTEM_PROMPT
        + f"\n\nFecha de hoy: {fecha_hoy}\n{perfil_linea}"
        + f"\nContexto RAG:\n---\n{contexto}\n---"
    )
    messages = [SystemMessage(content=system)] + state["messages"]
    respuesta = _invoke_con_retry(llm_con_tools, messages)
    return {"messages": [respuesta]}

def router_react(state: AgentStateReAct) -> _Literal["tools", "__end__"]:
    ultimo = state["messages"][-1]
    if hasattr(ultimo, "tool_calls") and ultimo.tool_calls:
        return "tools"
    return "__end__"

workflow_react = StateGraph(AgentStateReAct)
workflow_react.add_node("agente", nodo_agente_react)
workflow_react.add_node("tools",  nodo_tools)
workflow_react.add_edge(START, "agente")
workflow_react.add_conditional_edges("agente", router_react, {"tools": "tools", "__end__": END})
workflow_react.add_edge("tools", "agente")

agente_react = workflow_react.compile(checkpointer=MemorySaver())

# ── Prueba del agente ReAct ──────────────────────────────────────────────────
import uuid as _uuid
_cfg = {"configurable": {"thread_id": str(_uuid.uuid4())}}

preguntas_react = [
    "Soy autónomo, ¿qué tengo pendiente en los próximos 60 días?",
    "¿Cuál es el plazo del modelo 303?",
]
_state_react = {"messages": [], "perfil": "autonomo", "contexto_rag": ""}

for p in preguntas_react:
    _state_react["messages"] = _state_react.get("messages", []) + [HumanMessage(content=p)]
    _result = agente_react.invoke(_state_react, config=_cfg)
    _state_react = _result
    respuesta_react = next(
        (m.content for m in reversed(_result["messages"]) if isinstance(m, AIMessage) and m.content), ""
    )
    print(f"\nPregunta: {p}")
    print(f"Respuesta: {respuesta_react[:400]}...")

## 8. LLM-as-Judge — evaluación de calidad

Un segundo LLM evalúa cada respuesta del agente con puntuación estructurada en tres dimensiones: **precisión técnica**, **claridad** y **completitud**. Permite detectar sistemáticamente cuándo el agente responde mal sin revisar manualmente cada caso.

In [ ]:
import json
import time
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

PROMPT_JUEZ = ChatPromptTemplate.from_template("""
Eres un evaluador experto en asesoría fiscal española.
Evalúa la calidad de esta respuesta según los criterios dados.

**Pregunta:** {pregunta}
**Respuesta a evaluar:** {respuesta}
**Criterios:** {criterios}

Responde ÚNICAMENTE con este JSON (sin markdown ni bloques de código):
{{
  "puntuacion_global": número entre 1 y 10,
  "precision_tecnica": número entre 1 y 10,
  "claridad": número entre 1 y 10,
  "completitud": número entre 1 y 10,
  "justificacion": "máximo 2 oraciones"
}}
""")

def evaluar_respuesta(pregunta: str, respuesta: str) -> dict:
    """Evalúa una respuesta del agente con LLM-as-Judge. Devuelve dict con puntuaciones."""
    criterios = "precisión de fechas y plazos, claridad de la explicación, completitud según el perfil del usuario"
    messages = (PROMPT_JUEZ | StrOutputParser())
    # Construir los mensajes del prompt manualmente para pasar por _invoke_con_retry
    prompt_messages = PROMPT_JUEZ.format_messages(
        pregunta=pregunta, respuesta=respuesta, criterios=criterios
    )
    raw = _invoke_con_retry(llm, prompt_messages).content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

def evaluar_pipeline(pares_pregunta_respuesta: list[tuple[str, str]], nombre: str = "Agente") -> dict:
    """Evalúa un conjunto de pares pregunta/respuesta y devuelve métricas agregadas."""
    evaluaciones = []
    for pregunta, respuesta in pares_pregunta_respuesta:
        ev = evaluar_respuesta(pregunta, respuesta)
        ev["pregunta"] = pregunta
        evaluaciones.append(ev)
        print(f"  [{ev['puntuacion_global']}/10] {pregunta[:70]}")

    scores = [e["puntuacion_global"] for e in evaluaciones]
    return {
        "nombre":         nombre,
        "score_promedio": round(float(np.mean(scores)), 2),
        "score_min":      min(scores),
        "score_max":      max(scores),
        "evaluaciones":   evaluaciones,
    }

# ── Benchmark mínimo con respuestas reales del agente ───────────────────────
benchmark = [
    (
        "Soy autónomo, ¿cuándo presento el modelo 303 del 2T?",
        "**Perfil:** Autónomo.\n**Modelo 303 — IVA 2T 2026:**\n- Fecha límite: 20 de julio de 2026\n- Inicio de preparación: 10 de julio de 2026\n*Fuente: Calendario fiscal AEAT 2026*"
    ),
    (
        "¿Qué obligaciones tiene una sociedad en el segundo trimestre?",
        "**Perfil:** Sociedad.\nModelo 303, 111, 115 y 202 con plazo el 20 de julio.\n*Fuente: Mapa de obligaciones por perfil*"
    ),
    (
        "¿Cómo se calcula la casilla 01 del modelo 303?",
        "La casilla 01 recoge la base imponible de operaciones al tipo general del 21%.\n*Fuente: Manual práctico IVA 303 (AEAT 2025)*"
    ),
]

print("Evaluando respuestas con LLM-as-Judge...\n")
resultados = evaluar_pipeline(benchmark, nombre="Agente Fiscal v2")

print(f"\nScore promedio: {resultados['score_promedio']}/10")
print(f"Rango: {resultados['score_min']} – {resultados['score_max']}")
print("\nDetalle por pregunta:")
for ev in resultados["evaluaciones"]:
    print(f"\n  Pregunta: {ev['pregunta'][:80]}")
    print(f"  Global: {ev['puntuacion_global']} | Precisión: {ev['precision_tecnica']} | Claridad: {ev['claridad']} | Completitud: {ev['completitud']}")
    print(f"  {ev['justificacion']}")

## 9. Demo interactiva

Ejecuta la celda siguiente para abrir un chat con el agente. Escribe `salir` para terminar la sesión.

In [ ]:
import uuid

def chat(perfil_inicial: str = ""):
    """Sesión de chat interactiva con el agente fiscal."""
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    state = {"messages": [], "perfil": perfil_inicial, "contexto_rag": "", "tipo_consulta": ""}

    print("=" * 60)
    print("  ASISTENTE FISCAL — Gestoría España")
    print("=" * 60)
    if perfil_inicial:
        print(f"  Perfil activo: {perfil_inicial.upper()}")
    print("  Escribe 'salir' para terminar.\n")

    while True:
        pregunta = input("Tú: ").strip()
        if pregunta.lower() in ("salir", "exit", "quit"):
            print("Sesión finalizada.")
            break
        if not pregunta:
            continue

        state["messages"] = state.get("messages", []) + [HumanMessage(content=pregunta)]
        result = agente.invoke(state, config=config)
        state = result

        respuesta = result["messages"][-1].content
        print(f"\nAsistente: {respuesta}\n")
        print("-" * 60)

chat(perfil_inicial="")

---

### Casos de prueba documentados

Los 5 casos mínimos requeridos se prueban en las celdas siguientes (sin entrada interactiva).

In [ ]:
def preguntar(pregunta: str, state: dict, config: dict) -> tuple[str, dict]:
    state["messages"] = state.get("messages", []) + [HumanMessage(content=pregunta)]
    result = agente.invoke(state, config=config)
    return result["messages"][-1].content, result

# Sesión de prueba
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}
state = {"messages": [], "perfil": "", "contexto_rag": "", "tipo_consulta": ""}

casos = [
    # Caso 1 — Plazo de un modelo concreto → router: plazos
    "¿Cuál es el plazo para presentar el modelo 303 del primer trimestre de 2026?",
    # Caso 2 — Cómo rellenar una casilla → router: documentos
    "¿Cómo se calcula la casilla 01 del modelo 303?",
    # Caso 3 — Obligaciones autónomo 1T → router: plazos
    "Soy autónomo en estimación directa. ¿Qué declaraciones tengo que presentar en el primer trimestre?",
    # Caso 4 — Obligaciones sociedad 2T → router: plazos
    "Somos una sociedad limitada. ¿Qué obligaciones fiscales tenemos en el segundo trimestre?",
    # Caso 5 — Pregunta encadenada (memoria + antelación) → router: plazos
    "¿Y cuándo debería empezar a preparar esas declaraciones para llegar a tiempo?",
]

for i, pregunta in enumerate(casos, 1):
    print(f"\n{'='*60}")
    print(f"CASO {i}: {pregunta}")
    print('='*60)
    respuesta, state = preguntar(pregunta, state, config)
    print(f"RESPUESTA:\n{respuesta}")